In [3]:
# ============================================================
# TRANSFORM SPRINTS — LOCAL SILVER LAYER
# ============================================================

import os
from pyspark.sql import functions as F

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:08:11 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:08:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:08:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:08:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:08:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:08:11 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:08:11 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [5]:
# -----------------------------------------
# 2. Define Bronze + Silver paths
# -----------------------------------------
bronze_file = f"{BRONZE_ROOT}/sprints/sprints.parquet"
silver_folder = f"{SILVER_ROOT}/sprints"
silver_file = f"{silver_folder}/sprints_silver.parquet"

os.makedirs(silver_folder, exist_ok=True)

print("Bronze:", bronze_file)
print("Silver:", silver_file)

Bronze: /Users/manoharazalki/F1-ANALYTICS/data/bronze/sprints/sprints.parquet
Silver: /Users/manoharazalki/F1-ANALYTICS/data/silver/sprints/sprints_silver.parquet


In [6]:
# -----------------------------------------
# 3. Read Bronze Sprints
# -----------------------------------------
sprints_df = spark.read.parquet(bronze_file)
print("Bronze sprints loaded")
sprints_df.show(5, truncate=False)

Bronze sprints loaded
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull     |perez         |2   |17  |1

In [7]:
# -----------------------------------------
# 4. Select + rename columns (standardization)
# -----------------------------------------
sprints_selected_df = (
    sprints_df
        .select(
            "season",
            "round",
            "constructorId",
            "driverId",
            "date",
            "raceName",
            "grid",
            "laps",
            "number",
            "points",
            "position",
            "positionText",
            "status",
            "ingest_timestamp",
            "source_file"
        )
        .withColumnsRenamed(
            {
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "car_number",
                "position": "final_position",
                "positionText": "final_position_text"
            }
        )
)

print("Sprints renamed preview")
sprints_selected_df.show(5, truncate=False)

Sprints renamed preview
+------+-----+--------------+--------------+----------+---------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+-------------------------------------------------------------+
|season|round|constructor_id|driver_id     |race_date |race_name            |grid_position|completed_laps|car_number|points|final_position|final_position_text|status  |ingest_timestamp          |source_file                                                  |
+------+-----+--------------+--------------+----------+---------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+-------------------------------------------------------------+
|2023  |4    |red_bull      |perez         |2023-04-30|azerbaijan grand prix|2            |17            |11        |8.0   |1             |1                  |Finished|2026-06-08 21:50:12.355038|/Users/

In [8]:
# -----------------------------------------
# 5. Business key validation
# -----------------------------------------
sprints_valid_df = (
    sprints_selected_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

print("Invalid rows removed:", sprints_selected_df.count() - sprints_valid_df.count())

Invalid rows removed: 20


In [9]:
# -----------------------------------------
# 6. Title Case race_name
# -----------------------------------------
sprints_final_df = (
    sprints_valid_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

print("Sprints Silver preview")
sprints_final_df.show(5, truncate=False)

Sprints Silver preview


26/06/08 23:08:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+-----+--------------+----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+-------------------------------------------------------------+
|season|round|constructor_id|driver_id |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|final_position_text|status  |ingest_timestamp          |source_file                                                  |
+------+-----+--------------+----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+-------------------------------------------------------------+
|2021  |10   |alfa          |giovinazzi|2021-07-18|British Grand Prix|14           |17            |99        |0.0   |15            |15                 |Finished|2026-06-08 21:50:12.355038|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprin

In [10]:
# -----------------------------------------
# 7. Write Silver output (Option C)
# -----------------------------------------
(
    sprints_final_df
        .write
        .mode("overwrite")
        .parquet(silver_file)
)

print(f"Sprints Silver written to: {silver_file}")

Sprints Silver written to: /Users/manoharazalki/F1-ANALYTICS/data/silver/sprints/sprints_silver.parquet


In [11]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(silver_file).show(5, truncate=False)

+------+-----+--------------+----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+-------------------------------------------------------------+
|season|round|constructor_id|driver_id |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|final_position_text|status  |ingest_timestamp          |source_file                                                  |
+------+-----+--------------+----------+----------+------------------+-------------+--------------+----------+------+--------------+-------------------+--------+--------------------------+-------------------------------------------------------------+
|2021  |10   |alfa          |giovinazzi|2021-07-18|British Grand Prix|14           |17            |99        |0.0   |15            |15                 |Finished|2026-06-08 21:50:12.355038|/Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/sprin